In [1]:
import os
import sys
sys.path.append("../") # go to parent dir
%load_ext autoreload
%autoreload 2

In [24]:
from functions.loading_functions import *
train_df = get_dataset("train")
train_df

,Image_ID,class,confidence,ymin,xmin,ymax,xmax,class_id,ImagePath
0,ID_nBgcAR.jpg,healthy,1.0,75.0,15.0,162.0,195.0,2,dataset/images/train/ID_nBgcAR.jpg
1,ID_nBgcAR.jpg,healthy,1.0,58.0,1.0,133.0,171.0,2,dataset/images/train/ID_nBgcAR.jpg
2,ID_nBgcAR.jpg,healthy,1.0,42.0,29.0,377.0,349.0,2,dataset/images/train/ID_nBgcAR.jpg
3,ID_Kw2v8A.jpg,healthy,1.0,112.0,124.0,404.0,341.0,2,dataset/images/train/ID_Kw2v8A.jpg
4,ID_Kw2v8A.jpg,healthy,1.0,148.0,259.0,413.0,412.0,2,dataset/images/train/ID_Kw2v8A.jpg
...,...,...,...,...,...,...,...,...,...
9787,ID_WULBQy.jpeg,anthracnose,1.0,136.0,1440.0,1593.0,4000.0,0,dataset/images/train/ID_WULBQy.jpeg
9788,ID_WULBQy.jpeg,anthracnose,1.0,89.0,0.0,1174.0,1139.0,0,dataset/images/train/ID_WULBQy.jpeg
9789,ID_SVzl5X.jpeg,anthracnose,1.0,18.0,360.0,1800.0,2330.0,0,dataset/images/train/ID_SVzl5X.jpeg
9790,ID_xDTIEp.jpeg,anthracnose,1.0,736.0,174.0,2691.0,4032.0,0,dataset/images/train/ID_xDTIEp.jpeg


In [46]:
train_df

,Image_ID,class,confidence,ymin,xmin,ymax,xmax,class_id,ImagePath
0,ID_nBgcAR.jpg,healthy,1.0,75.0,15.0,162.0,195.0,2,dataset/images/train/ID_nBgcAR.jpg
1,ID_nBgcAR.jpg,healthy,1.0,58.0,1.0,133.0,171.0,2,dataset/images/train/ID_nBgcAR.jpg
2,ID_nBgcAR.jpg,healthy,1.0,42.0,29.0,377.0,349.0,2,dataset/images/train/ID_nBgcAR.jpg
3,ID_Kw2v8A.jpg,healthy,1.0,112.0,124.0,404.0,341.0,2,dataset/images/train/ID_Kw2v8A.jpg
4,ID_Kw2v8A.jpg,healthy,1.0,148.0,259.0,413.0,412.0,2,dataset/images/train/ID_Kw2v8A.jpg
...,...,...,...,...,...,...,...,...,...
9787,ID_WULBQy.jpeg,anthracnose,1.0,136.0,1440.0,1593.0,4000.0,0,dataset/images/train/ID_WULBQy.jpeg
9788,ID_WULBQy.jpeg,anthracnose,1.0,89.0,0.0,1174.0,1139.0,0,dataset/images/train/ID_WULBQy.jpeg
9789,ID_SVzl5X.jpeg,anthracnose,1.0,18.0,360.0,1800.0,2330.0,0,dataset/images/train/ID_SVzl5X.jpeg
9790,ID_xDTIEp.jpeg,anthracnose,1.0,736.0,174.0,2691.0,4032.0,0,dataset/images/train/ID_xDTIEp.jpeg


In [29]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil
import yaml
import torch

# Step 1: Process the original dataset and convert to YOLO format
def process_dataset(df, output_dir='yolo_dataset'):
    """
    Convert the original dataset format to YOLO format.
    
    Args:
        df: pandas DataFrame with columns ['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax', 'class_id', 'ImagePath']
        output_dir: directory to save the YOLO format dataset
    """
    # Create necessary directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)
    
    # Get unique classes and create class mapping
    classes = df['class'].unique().tolist()
    class_to_id = {class_name: i for i, class_name in enumerate(classes)}
    
    # Save the class mapping to a file
    with open(os.path.join(output_dir, 'classes.txt'), 'w') as f:
        for class_name in classes:
            f.write(f"{class_name}\n")
    
    # Process each image
    processed_images = set()
    for _, row in df.iterrows():
        img_path = row['ImagePath']
        img_id = row['Image_ID']
        
        # Skip if we've already processed this image
        if img_id in processed_images:
            continue
        
        # Copy the image to the YOLO dataset
        dst_img_path = os.path.join(output_dir, 'images', img_id)
        shutil.copy(img_path, dst_img_path)
        
        # Create a label file for this image
        label_filename = img_id.split(".")[0]+".txt"
        label_path = os.path.join(output_dir, 'labels', label_filename)
        
        # Get all annotations for this image
        img_annotations = df[df['Image_ID'] == img_id].copy()
        
        # Open the image to get dimensions
        with Image.open(img_path) as img:
            img_width, img_height = img.size
        
        # Write annotations in YOLO format
        with open(label_path, 'w') as f:
            for _, ann in img_annotations.iterrows():
                # Convert bbox coordinates to YOLO format (normalized center x, center y, width, height)
                x_min, y_min = ann['xmin'], ann['ymin']
                x_max, y_max = ann['xmax'], ann['ymax']
                
                # Normalize coordinates
                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width = (x_max - x_min) / img_width
                height = (y_max - y_min) / img_height
                
                # Get class ID
                class_id = class_to_id[ann['class']]
                
                # In case they are none
                # Write in YOLO format: class_id x_center y_center width height
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
        
        processed_images.add(img_id)
    
    print(f"Processed {len(processed_images)} images with {len(df)} annotations.")
    return classes

def split_dataset(output_dir='yolo_dataset', train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    """
    Split the dataset into train, validation, and test sets.
    
    Args:
        output_dir: directory containing the YOLO format dataset
        train_ratio: ratio of training data
        val_ratio: ratio of validation data
        test_ratio: ratio of test data
    """
    # Get all image filenames
    image_dir = os.path.join(output_dir, 'images')
    all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Create train, val, test directories
    for split in ['train', 'val', 'test']:
        for subdir in ['images', 'labels']:
            os.makedirs(os.path.join(output_dir, split, subdir), exist_ok=True)
    
    # Split the dataset
    train_images, temp_images = train_test_split(all_images, train_size=train_ratio, random_state=42)
    val_size = val_ratio / (val_ratio + test_ratio)
    val_images, test_images = train_test_split(temp_images, train_size=val_size, random_state=42)
    
    # Move files to respective directories
    for split, images in [('train', train_images), ('val', val_images), ('test', test_images)]:
        for img_file in images:
            # Get corresponding label file
            label_file = os.path.splitext(img_file)[0] + '.txt'
            
            # Move image
            src_img = os.path.join(output_dir, 'images', img_file)
            dst_img = os.path.join(output_dir, split, 'images', img_file)
            shutil.copy(src_img, dst_img)
            
            # Move label
            src_label = os.path.join(output_dir, 'labels', label_file)
            dst_label = os.path.join(output_dir, split, 'labels', label_file)
            if os.path.exists(src_label):  # Some images might not have annotations
                shutil.copy(src_label, dst_label)
    
    print(f"Dataset split: {len(train_images)} train, {len(val_images)} validation, {len(test_images)} test images.")

def create_yaml_config(classes, output_dir='yolo_dataset'):
    """
    Create a YAML configuration file for YOLOv5.
    
    Args:
        classes: list of class names
        output_dir: directory containing the YOLO format dataset
    """
    config = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(classes),
        'names': classes
    }
    
    with open(os.path.join(output_dir, 'dataset.yaml'), 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"Created YAML configuration file at {os.path.join(output_dir, 'dataset.yaml')}")

# Step 2: Set up and train the YOLO model (YOLOv11)
def train_yolov11(dataset_yaml, model_name_path: str,  model_size='s', epochs=100, batch_size=16, image_size=640):
    """
    Train a YOLOv11 model.
    
    Args:
        dataset_yaml: path to the YAML configuration file
        model_size: YOLOv11 model size ('n', 's', 'm', 'l', 'x')
        epochs: number of training epochs
        batch_size: batch size
        image_size: input image size
    """
    # Install Ultralytics package (which includes YOLOv11)
    os.system('pip install ultralytics')
    
    # Start training with YOLOv11
    import ultralytics
    from ultralytics import YOLO
    
    # Print Ultralytics version for reference
    print(f"Ultralytics version: {ultralytics.__version__}")
    
    # Load the base model
    model = YOLO(model_name_path)
    
    # Train the model using the dataset YAML config
    model.train(
        data=dataset_yaml,
        epochs=epochs,
        batch=batch_size,
        device=[0],  # List all GPU indices to use
        imgsz=image_size,
        patience=50,  # Early stopping patience
        cache=True,
        project='yolov11_plant_detection',
        name=f'yolov11{model_size}_run1',
        save=True,    # Save best model
        pretrained=True,
        verbose=True
    )

# Step 3: Run inference on test images without bounding boxes
def run_inference(model_path, test_image_dir, output_dir='predictions', conf_threshold=0.25):
    """
    Run inference on test images using YOLOv11.
    
    Args:
        model_path: path to the trained YOLOv11 model
        test_image_dir: directory containing test images
        output_dir: directory to save prediction results
        conf_threshold: confidence threshold for detections
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load model using Ultralytics YOLO
    from ultralytics import YOLO
    model = YOLO(model_path)
    
    # Set confidence threshold
    # Note: In YOLOv11, this is done at prediction time
    
    # Run inference
    results = model.predict(
        source=test_image_dir,
        conf=conf_threshold,
        save=True,
        save_txt=True,
        save_conf=True,
        project=output_dir,
        name='detect',
        verbose=True
    )
    
    # Export results to a CSV file
    predictions = []
    
    for result in results:
        img_name = os.path.basename(result.path)
        boxes = result.boxes
        
        if len(boxes) > 0:
            # Extract coordinates, confidence, and class
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()  # xyxy format (top-left, bottom-right)
                conf = box.conf.item()
                cls = int(box.cls.item())
                class_name = model.names[cls]
                
                predictions.append({
                    'image_name': img_name,
                    'class': class_name,
                    'confidence': conf,
                    'xmin': x1,
                    'ymin': y1,
                    'xmax': x2,
                    'ymax': y2
                })
    
    # Create DataFrame and save
    if predictions:
        pred_df = pd.DataFrame(predictions)
        pred_df.to_csv(os.path.join(output_dir, 'predictions.csv'), index=False)
    
    print(f"Inference complete. Results saved to {output_dir}")

# Step 4: Aggregate image-level predictions for multi-label evaluation
def aggregate_predictions(predictions_csv, output_csv='image_level_predictions.csv'):
    """
    Aggregate bounding box predictions to image-level class predictions.
    
    Args:
        predictions_csv: path to CSV file with bounding box predictions
        output_csv: path to save image-level predictions
    """
    # Load predictions
    pred_df = pd.read_csv(predictions_csv)
    
    # Group by image and aggregate classes
    image_level = pred_df.groupby('image_name')['class'].apply(lambda x: list(set(x))).reset_index()
    image_level.rename(columns={'class': 'predicted_classes'}, inplace=True)
    
    # Save to CSV
    image_level.to_csv(output_csv, index=False)
    
    print(f"Aggregated predictions saved to {output_csv}")


In [30]:
# Process dataset
classes = process_dataset(train_df)

Processed 5529 images with 9792 annotations.


In [32]:
split_dataset()

Dataset split: 3672 train, 1050 validation, 525 test images.


In [33]:
create_yaml_config(classes)

Created YAML configuration file at yolo_dataset/dataset.yaml


In [34]:
# Train model using YOLOv11
train_yolov11(dataset_yaml='yolo_dataset/dataset.yaml', model_name_path='yolo11s.pt', model_size='s', epochs=20, batch_size=64)

Ultralytics version: 8.3.94
Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: task=detect, mode=train, model=yolo11s.pt, data=yolo_dataset/dataset.yaml, epochs=20, time=None, patience=50, batch=64, imgsz=640, save=True, save_period=-1, cache=True, device=[0], workers=8, project=yolov11_plant_detection, name=yolov11s_run12, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=F

train: Scanning /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/labels... 3672 images, 0 backgrounds, 599 corrupt: 100%|██████████| 3672/3672 [00:09<00:00, 379.41it/s]

train: WARNING ⚠️ /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/images/ID_AC3jGA.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2288      1.0926]
train: WARNING ⚠️ /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/images/ID_AIHFIo.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.3267      1.1366]
train: WARNING ⚠️ /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/images/ID_AJD939.jpg: corrupt JPEG restored and saved
train: WARNING ⚠️ /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/images/ID_AK7dFo.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.3218      1.2725]
train: WARNING ⚠️ /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/images/ID_AMshRT.jpeg:

train: New cache created: /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/train/labels.cache
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (2.7GB RAM): 100%|██████████| 3073/3073 [00:14<00:00, 216.71it/s]
val: Scanning /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolo_dataset/val/labels... 1050 images, 0 backgrounds, 197 corrupt: 100%|██████████| 1050/1050 [00:05<00:00, 196.53it/s]

...

      20/20      18.9G      1.107     0.9789      1.524          3        640: 100%|██████████| 49/49 [00:15<00:00,  3.18it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:02<00:00,  2.44it/s]

                   all        853       1447      0.737      0.646      0.698       0.47



20 epochs completed in 0.108 hours.
Optimizer stripped from yolov11_plant_detection/yolov11s_run12/weights/last.pt, 19.2MB
Optimizer stripped from yolov11_plant_detection/yolov11s_run12/weights/best.pt, 19.2MB

Validating yolov11_plant_detection/yolov11s_run12/weights/best.pt...
Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
YOLO11s summary (fused): 100 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:04<00:00,  1.69it/s]


                   all        853       1447      0.737      0.646      0.698      0.469
               healthy        223        545      0.689      0.594      0.656      0.421
           anthracnose        219        330      0.713       0.63      0.663      0.433
                 cssvd        412        572      0.809      0.713      0.775      0.554
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to yolov11_plant_detection/yolov11s_run12


In [35]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from pathlib import Path

def plot_yolo_performance(results_dir, output_dir='performance_plots'):
    """
    Generate comprehensive performance plots for YOLOv11 training results.
    
    Args:
        results_dir: Directory containing YOLO training results
        output_dir: Directory to save performance plots
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Check if it's Ultralytics format (YOLOv8/YOLOv11) or older YOLOv5 format
    results_csv = os.path.join(results_dir, 'results.csv')
    
    if os.path.exists(results_csv):
        # Newer Ultralytics format
        df = pd.read_csv(results_csv)
        
        # Check if we have precision-recall data
        if all(col in df.columns for col in ['precision', 'recall', 'mAP50', 'mAP50-95']):
            # Create figure with multiple subplots
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle('YOLO Training Performance Metrics', fontsize=16)
            
            # Plot Loss curves
            ax = axes[0, 0]
            if 'box_loss' in df.columns and 'cls_loss' in df.columns:
                ax.plot(df['epoch'], df['box_loss'], label='Box Loss')
                ax.plot(df['epoch'], df['cls_loss'], label='Class Loss')
                if 'dfl_loss' in df.columns:  # YOLOv8/v11 specific
                    ax.plot(df['epoch'], df['dfl_loss'], label='DFL Loss')
            else:
                ax.plot(df['epoch'], df['train_loss'], label='Train Loss')
                if 'val_loss' in df.columns:
                    ax.plot(df['epoch'], df['val_loss'], label='Val Loss')
            
            ax.set_title('Training Losses')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot Precision, Recall
            ax = axes[0, 1]
            ax.plot(df['epoch'], df['precision'], label='Precision')
            ax.plot(df['epoch'], df['recall'], label='Recall')
            ax.set_title('Precision and Recall')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Value')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot mAP values
            ax = axes[1, 0]
            ax.plot(df['epoch'], df['mAP50'], label='mAP@0.5')
            ax.plot(df['epoch'], df['mAP50-95'], label='mAP@0.5:0.95')
            ax.set_title('Mean Average Precision (mAP)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('mAP')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot learning rate if available
            ax = axes[1, 1]
            if 'lr0' in df.columns:
                ax.plot(df['epoch'], df['lr0'], label='Learning Rate')
                ax.set_title('Learning Rate Schedule')
                ax.set_xlabel('Epoch')
                ax.set_ylabel('Learning Rate')
                ax.legend()
                ax.grid(True, linestyle='--', alpha=0.6)
            else:
                # If learning rate is not available, plot a confusion matrix if available
                confusion_matrix_file = Path(results_dir) / 'confusion_matrix.png'
                if confusion_matrix_file.exists():
                    confusion_img = plt.imread(confusion_matrix_file)
                    ax.imshow(confusion_img)
                    ax.set_title('Confusion Matrix')
                    ax.axis('off')
                else:
                    # If no confusion matrix, show fitness
                    if 'fitness' in df.columns:
                        ax.plot(df['epoch'], df['fitness'], 'g-', label='Fitness')
                        ax.set_title('Model Fitness')
                        ax.set_xlabel('Epoch')
                        ax.set_ylabel('Fitness Value')
                        ax.legend()
                        ax.grid(True, linestyle='--', alpha=0.6)
            
            # Adjust layout and save
            plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for main title
            plt.savefig(os.path.join(output_dir, 'training_metrics.png'), dpi=300)
            plt.close()
            
            # Create PR curve plot if we have confidence data
            pr_curve_file = Path(results_dir) / 'PR_curve.png'
            if pr_curve_file.exists():
                pr_img = plt.imread(pr_curve_file)
                plt.figure(figsize=(10, 8))
                plt.imshow(pr_img)
                plt.axis('off')
                plt.title('Precision-Recall Curve')
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, 'pr_curve.png'), dpi=300)
                plt.close()
            
            # Create F1 score vs confidence threshold plot if available
            f1_curve_file = Path(results_dir) / 'F1_curve.png'
            if f1_curve_file.exists():
                f1_img = plt.imread(f1_curve_file)
                plt.figure(figsize=(10, 8))
                plt.imshow(f1_img)
                plt.axis('off')
                plt.title('F1 Score vs Confidence Threshold')
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, 'f1_curve.png'), dpi=300)
                plt.close()
            
            # Print final metrics
            final_epoch = df.iloc[-1]
            print("\nFinal Training Metrics:")
            print(f"Precision: {final_epoch['precision']:.4f}")
            print(f"Recall: {final_epoch['recall']:.4f}")
            print(f"mAP@0.5: {final_epoch['mAP50']:.4f}")
            print(f"mAP@0.5:0.95: {final_epoch['mAP50-95']:.4f}")
            
            return {
                'precision': final_epoch['precision'],
                'recall': final_epoch['recall'],
                'mAP50': final_epoch['mAP50'],
                'mAP50-95': final_epoch['mAP50-95']
            }
        else:
            print("Warning: CSV file does not contain expected metrics columns.")
    
    # Check for older YOLOv5 format results.txt
    results_txt = os.path.join(results_dir, 'results.txt')
    if os.path.exists(results_txt):
        # Parse results.txt in YOLOv5 format
        epochs, box_loss, obj_loss, cls_loss = [], [], [], []
        precision, recall, map50, map = [], [], [], []
        
        with open(results_txt, 'r') as f:
            for line in f:
                if line.startswith('  Epoch'):
                    continue  # Skip header line
                try:
                    # Extract metrics (format may vary)
                    parts = line.strip().split()
                    if len(parts) >= 12:
                        epoch = int(parts[0])
                        epochs.append(epoch)
                        
                        # Extract losses
                        box_loss.append(float(parts[2]))
                        obj_loss.append(float(parts[3]))
                        cls_loss.append(float(parts[4]))
                        
                        # Extract precision, recall, mAP
                        precision.append(float(parts[8]))
                        recall.append(float(parts[9]))
                        map50.append(float(parts[10]))
                        map.append(float(parts[11]))
                except Exception as e:
                    print(f"Error parsing line: {line}")
                    print(f"Exception: {e}")
        
        if epochs:
            # Create figure with multiple subplots
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle('YOLO Training Performance Metrics', fontsize=16)
            
            # Plot Loss curves
            ax = axes[0, 0]
            ax.plot(epochs, box_loss, label='Box Loss')
            ax.plot(epochs, obj_loss, label='Objectness Loss')
            ax.plot(epochs, cls_loss, label='Class Loss')
            ax.set_title('Training Losses')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot Precision, Recall
            ax = axes[0, 1]
            ax.plot(epochs, precision, label='Precision')
            ax.plot(epochs, recall, label='Recall')
            ax.set_title('Precision and Recall')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Value')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot mAP values
            ax = axes[1, 0]
            ax.plot(epochs, map50, label='mAP@0.5')
            ax.plot(epochs, map, label='mAP@0.5:0.95')
            ax.set_title('Mean Average Precision (mAP)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('mAP')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot combined loss
            ax = axes[1, 1]
            total_loss = [b + o + c for b, o, c in zip(box_loss, obj_loss, cls_loss)]
            ax.plot(epochs, total_loss, label='Total Loss')
            ax.set_title('Total Training Loss')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Adjust layout and save
            plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for main title
            plt.savefig(os.path.join(output_dir, 'training_metrics.png'), dpi=300)
            plt.close()
            
            # Print final metrics
            print("\nFinal Training Metrics:")
            print(f"Precision: {precision[-1]:.4f}")
            print(f"Recall: {recall[-1]:.4f}")
            print(f"mAP@0.5: {map50[-1]:.4f}")
            print(f"mAP@0.5:0.95: {map[-1]:.4f}")
            
            return {
                'precision': precision[-1],
                'recall': recall[-1],
                'mAP50': map50[-1],
                'mAP50-95': map[-1]
            }

In [37]:
plot_yolo_performance('/home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolov11_plant_detection/yolov11s_run12', output_dir='performance_plots')

In [39]:
test_image_dir = '/home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test'  # Your test images directory
model_path = '/home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/yolov11_plant_detection/yolov11s_run12/weights/best.pt'
# Run inference on test images
run_inference(model_path, test_image_dir)



WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1626 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_A16nzu.jpg: 640x480 1 healthy, 70.8ms
image 2/1626 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_A1Euyz.jpg: 640x480 1 anthracnose, 11.1ms
image 3/1626 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_A1HcV0.jpeg: 640x480 1 healthy, 1 anthracnose,

In [41]:
# Aggregate predictions for multi-label evaluation
aggregate_predictions('/home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/predictions/predictions.csv')

Aggregated predictions saved to image_level_predictions.csv


In [ ]:
Image_ID,class,confidence,ymin,xmin,ymax,xmax
ID_Genxyu.jpg,healthy,0.5,100,100,100,100

In [ ]:

# Example usage
if __name__ == "__main__":
    # Prepare test data
    gt_csv = prepare_test_data("test_data.csv", "test_images")
    
    # Visualize predictions
    batch_visualize_predictions("test_images", "predictions/predictions.csv", "visualizations")
    
    # Prepare submission
    submission_csv = prepare_submission("predictions/predictions.csv", "test_images", "submission.csv")
    
    # Evaluate predictions
    from evaluation_script import evaluate_multilabel
    evaluate_multilabel(gt_csv, submission_csv)

In [54]:
TEST_IMAGES_DIR

PosixPath('dataset/images/test')

In [48]:
from ultralytics import YOLO
from tqdm.notebook import tqdm

# Load the trained YOLO model
model = YOLO(model_path)

# Path to the test images directory
test_dir_path = TEST_IMAGES_DIR

# Get a list of all image files in the test directory
image_files = os.listdir(test_dir_path)

# Initialize an empty list to store the results for all images
all_data = []

# Initialize an empty list to store the results for all images
all_data = []

# Iterate through each image in the directory
for image_file in tqdm(image_files):
    # Full path to the image
    img_path = os.path.join(test_dir_path, image_file)

    # Make predictions on the image
    results = model(img_path)

    # Extract bounding boxes, confidence scores, and class labels
    boxes = results[0].boxes.xyxy.tolist() if results[0].boxes else []  # Bounding boxes in xyxy format
    classes = results[0].boxes.cls.tolist() if results[0].boxes else []  # Class indices
    confidences = results[0].boxes.conf.tolist() if results[0].boxes else []  # Confidence scores
    names = results[0].names  # Class names dictionary

    if boxes:  # If detections are found
        for box, cls, conf in zip(boxes, classes, confidences):
            x1, y1, x2, y2 = box
            detected_class = names[int(cls)]  # Get the class name from the names dictionary

            # Add the result to the all_data list
            all_data.append({
                'Image_ID': str(image_file),
                'class': detected_class,
                'confidence': conf,
                'ymin': y1,
                'xmin': x1,
                'ymax': y2,
                'xmax': x2
            })
    else:  # If no objects are detected
        all_data.append({
            'Image_ID': str(image_file),
            'class': "None",
            'confidence': None,
            'ymin': None,
            'xmin': None,
            'ymax': None,
            'xmax': None
        })



  0%|          | 0/1626 [00:00<?, ?it/s]


image 1/1 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_zSiC0w.jpg: 640x480 1 healthy, 9.4ms
Speed: 2.9ms preprocess, 9.4ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_kvMeX2.JPG: 640x480 1 cssvd, 10.8ms
Speed: 2.7ms preprocess, 10.8ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_EBhZzJ.jpeg: 640x640 2 anthracnoses, 8.8ms
Speed: 3.5ms preprocess, 8.8ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/jason-server/Documents/deep_learning_aueb/amini_cocoa_contamination/dataset/images/test/ID_p4aJv5.jpg: 640x480 2 cssvds, 10.0ms
Speed: 1.8ms preprocess, 10.0ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/jason-server

In [50]:
# Convert the list to a DataFrame for all images
sub = pd.DataFrame(all_data)

In [53]:
sub

,Image_ID,class,confidence,ymin,xmin,ymax,xmax
0,ID_zSiC0w.jpg,healthy,0.803784,911.792542,548.785400,2914.808594,1705.729614
1,ID_kvMeX2.JPG,cssvd,0.473394,602.720154,398.513397,2853.851807,1585.934692
2,ID_EBhZzJ.jpeg,anthracnose,0.819010,3.978644,311.530884,3010.936279,2203.006348
3,ID_EBhZzJ.jpeg,anthracnose,0.264222,31.505287,0.000000,1919.579956,1121.500732
4,ID_p4aJv5.jpg,cssvd,0.845414,220.992050,0.608660,1044.002808,436.094330
...,...,...,...,...,...,...,...
2794,ID_xjnuvj.jpg,healthy,0.756919,38.240887,98.341141,355.820465,334.627960
2795,ID_xjnuvj.jpg,healthy,0.619615,0.000000,184.657120,115.048332,403.408936
2796,ID_xjnuvj.jpg,healthy,0.492850,48.768326,10.391272,272.033508,177.581467
2797,ID_xjnuvj.jpg,healthy,0.300794,0.000000,122.022362,57.127178,252.787720


In [49]:
INPUT_DATA_DIR

PosixPath('.')

In [52]:
# Create submission file to be uploaded to Zindi for scoring
sub.to_csv(f'{INPUT_DATA_DIR / "BenchmarkSubmission.csv"}', index = False)